# Lesson 5: Layers and regions

**You will learn:** how to record several layers, decode information from them on held-out images, and compare a trained network with the same network untrained.
**Requires:** CPU only, about a minute.

**The question:** can animals be told from vehicles from each layer, and how much of that comes from training? An untrained network with the same architecture is the null: images differ in colour and texture, so even random features can separate some categories.

In [1]:
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)   # no progress bars
warnings.filterwarnings('ignore')                            # keep the lesson output short

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images (CIFAR-10 test set, 12 per category), committed with the lessons.
ASSETS = Path('../assets/lessons')
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [3]:
import torch
import torchvision
from PIL import Image

# SqueezeNet 1.1 trained on ImageNet (committed, 5 MB). It never saw these images.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()

TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files to a (batch, 3, 224, 224) array, as the network was trained."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',
        intervention=build_pytorch_ablation_fn(network))

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(c for c in subject.out_channels if not c.startswith('behavior:'))

['behavior', 'neural:IT', 'neural:V1', 'neural:V4', 'perturbation']

In [5]:
torch.manual_seed(0)
untrained = torchvision.models.squeezenet1_1().eval()          # same architecture, random weights
null_subject = squeezenet_subject(untrained, 'squeezenet1_1-untrained')

## Decode from each layer

Five-fold cross-validation with a regularized logistic classifier: every image is scored by a classifier that never saw it.

In [6]:
from brainscore_core.streaming import BufferedSession
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

regions = ['V1', 'V4', 'IT']
y = np.asarray(stimuli['label'])
folds = StratifiedKFold(5, shuffle=True, random_state=0)
decoder = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=3000))

rows = []
for name, s in (('trained', subject), ('untrained', null_subject)):
    session = BufferedSession(stimulus_set=stimuli, requested_output_channels=[f'neural:{r}' for r in regions])
    s.participate(session)
    for region in regions:
        X = np.asarray(session.collect(f'neural:{region}').values)
        rows.append({'network': name, 'region': region, 'units': X.shape[1],
                     'accuracy': cross_val_score(decoder, X, y, cv=folds).mean()})
table = pd.DataFrame(rows).pivot(index='region', columns='network', values='accuracy').loc[regions]
table['trained minus untrained'] = table['trained'] - table['untrained']
table.round(3)

network,trained,untrained,trained minus untrained
region,,,
V1,0.808,0.717,0.092
V4,0.875,0.800,0.075
IT,0.867,0.808,0.058


Chance is 0.6 (72 animals, 48 vehicles). Read the last column: the part of each layer's accuracy that training added. Whether it grows from V1 to IT is the answer to the question, and it need not.

## Summary

- One session records several regions in one pass.
- Decode on held-out images, and compare with an untrained network of the same architecture.
- The region map is a hypothesis; this kind of comparison is how you test it.

**Next:** [Lesson 6](06_behavior.ipynb) turns a layer into answers.